# Relaciones del modelo relacional `core`

Este documento resume las relaciones principales del modelo relacional construido en el esquema `core` para el proyecto de análisis logístico del dataset Olist.

El objetivo de este modelo es conservar la estructura original del dataset, mantener la trazabilidad entre entidades y preparar una base limpia sobre la que posteriormente se construirá la capa analítica dimensional.

## Representación gráfica del modelo

![Esquema relacional del modelo core](../outputs/modelo_relacional_tansaccional.png)

## Tablas principales

El esquema `core` contiene las siguientes tablas:

- `core.geolocations`
- `core.customers`
- `core.sellers`
- `core.products`
- `core.orders`
- `core.order_items`
- `core.order_payments`
- `core.order_reviews`

La tabla `orders` actúa como eje del ciclo de pedido, mientras que `order_items` permite conectar pedidos, productos y vendedores a nivel de línea de pedido.

## Relaciones principales

### `geolocations` → `customers`

    core.geolocations 1 ─── N core.customers

Un mismo prefijo postal puede estar asociado a muchos clientes. Cada cliente tiene un único prefijo postal.

Clave foránea:

    core.customers.customer_zip_code_prefix
    → core.geolocations.geolocation_zip_code_prefix

### `geolocations` → `sellers`

    core.geolocations 1 ─── N core.sellers

Un mismo prefijo postal puede estar asociado a muchos vendedores. Cada vendedor tiene un único prefijo postal.

Clave foránea:

    core.sellers.seller_zip_code_prefix
    → core.geolocations.geolocation_zip_code_prefix

### `customers` → `orders`

    core.customers 1 ─── N core.orders

Un cliente puede tener varios pedidos. Cada pedido pertenece a un único cliente.

Clave foránea:

    core.orders.customer_id
    → core.customers.customer_id

### `orders` → `order_items`

    core.orders 1 ─── N core.order_items

Un pedido puede contener varias líneas de pedido. Cada línea de pedido pertenece a un único pedido.

Clave foránea:

    core.order_items.order_id
    → core.orders.order_id

### `orders` → `order_payments`

    core.orders 1 ─── N core.order_payments

Un pedido puede tener uno o varios registros de pago.

Clave foránea:

    core.order_payments.order_id
    → core.orders.order_id

### `orders` → `order_reviews`

    core.orders 1 ─── N core.order_reviews

Un pedido puede tener una reseña, ninguna reseña o más de una reseña.

Clave foránea:

    core.order_reviews.order_id
    → core.orders.order_id

### `products` → `order_items`

    core.products 1 ─── N core.order_items

Un producto puede aparecer en muchas líneas de pedido. Cada línea de pedido hace referencia a un único producto.

Clave foránea:

    core.order_items.product_id
    → core.products.product_id

### `sellers` → `order_items`

    core.sellers 1 ─── N core.order_items

Un vendedor puede aparecer en muchas líneas de pedido. Cada línea de pedido está asociada a un único vendedor.

Clave foránea:

    core.order_items.seller_id
    → core.sellers.seller_id

## Relaciones conceptuales muchos a muchos

Algunas relaciones del negocio son conceptualmente muchos a muchos, pero quedan resueltas mediante la tabla `core.order_items`.

### Pedidos y productos

    core.orders N ─── M core.products

Un pedido puede contener varios productos y un producto puede aparecer en muchos pedidos.

Esta relación se resuelve así:

    core.orders 1 ─── N core.order_items N ─── 1 core.products

### Pedidos y vendedores

    core.orders N ─── M core.sellers

Un pedido puede contener productos de varios vendedores y un vendedor puede aparecer en muchos pedidos.

Esta relación se resuelve así:

    core.orders 1 ─── N core.order_items N ─── 1 core.sellers

## Orden lógico de creación de tablas

El orden de creación de las tablas depende de las claves foráneas. Primero se crean las tablas que no dependen de otras y después las tablas que apuntan a ellas.

Orden utilizado:

    1. core.geolocations
    2. core.customers
    3. core.sellers
    4. core.products
    5. core.orders
    6. core.order_items
    7. core.order_payments
    8. core.order_reviews

Este orden permite crear las relaciones sin errores de dependencias.

## Resumen de claves foráneas

    core.customers.customer_zip_code_prefix
    → core.geolocations.geolocation_zip_code_prefix

    core.sellers.seller_zip_code_prefix
    → core.geolocations.geolocation_zip_code_prefix

    core.orders.customer_id
    → core.customers.customer_id

    core.order_items.order_id
    → core.orders.order_id

    core.order_items.product_id
    → core.products.product_id

    core.order_items.seller_id
    → core.sellers.seller_id

    core.order_payments.order_id
    → core.orders.order_id

    core.order_reviews.order_id
    → core.orders.order_id

## Función del modelo `core`

El modelo `core` funciona como capa relacional limpia del proyecto. Su función no es todavía calcular indicadores finales, sino conservar las entidades principales del dataset, sus relaciones y sus distintos niveles de granularidad.

Sobre esta capa se construirá posteriormente el esquema `analytics`, donde se desarrollará la constelación analítica ligera con tablas de hechos y dimensiones orientadas al análisis logístico.